# MVNX Parquet 시계열 데이터 검증 노트북
이 노트북은 전처리된 Parquet 파일들의 무결성, 컬럼 구조, 메타데이터 주입 여부를 자동으로 탐색하고 리포팅하는 재사용 가능한 함수 컴포넌트들을 제공합니다.

In [2]:
import pandas as pd
from pathlib import Path
import os

In [3]:
def verify_parquet_file(file_path):
    """
    단일 Parquet 파일을 열어 메타데이터 및 시계열 결측치, Shape를 검증합니다.
    """
    path_obj = Path(file_path)
    print(f"\n{'='*50}")
    print(f"🔍 [분석 시작]: {path_obj.name}")
    
    # 1. 파일 로딩 속도 및 Shape
    df = pd.read_parquet(path_obj)
    print(f"✅ Data Shape: {df.shape[0]:,} Rows × {df.shape[1]:,} Columns")
    
    # 2. 필수 메타데이터 컬럼 검증
    meta_cols = ['time_ms', 'group', 'subject_id', 'speed', 'file_name']
    missing_meta = [c for c in meta_cols if c not in df.columns]
    if missing_meta:
        print(f"⚠️ 누락된 메타데이터 컬럼: {missing_meta}")
    else:
        print(f"✅ 필수 메타데이터({len(meta_cols)}종) 정상 마운트")
        
    # 3. 고유 속성군 분포 (Subject, Speed 종류 파악)
    if 'subject_id' in df.columns:
        print(f"👤 포함된 피험자 종류({df['subject_id'].nunique()}명): {df['subject_id'].unique()[:5].tolist()}...")
    if 'speed' in df.columns:
        print(f"🏃‍♂️ 포함된 보행속도 분포: {df['speed'].unique().tolist()}")
        
    # 4. 시계열 데이터(Features) 결측값(NaN) 요약
    feature_cols = [c for c in df.columns if c not in meta_cols]
    na_counts = df[feature_cols].isna().sum()
    cols_with_na = na_counts[na_counts > 0]
    if len(cols_with_na) > 0:
        print(f"⚠️ 결측치(NaN)가 포함된 피처 개수: {len(cols_with_na)}개")
    else:
        print("✅ 시계열 부분 결측치 완전 무결 (Clean Data)")
        
    return df

In [12]:
# 화면에 출력할 행(Row, 세로줄) 개수에 제한을 두지 않음
pd.set_option('display.max_rows', None)

# 화면에 출력할 열(Column, 가로줄) 개수에 제한을 두지 않음
pd.set_option('display.max_columns', None)

# 한 셀(칸) 안에 들어가는 글자 수가 길어도 절대 자르지 않음
pd.set_option('display.max_colwidth', None)

# 노트북 위치를 기반으로 상대경로 설정 (Soft-coding)
PROCESSED_DIR = Path("../data/processed").resolve()

# 🪣 1. 빈 데이터 '리스트' 바구니를 미리 준비합니다 (빈 데이터프레임 X)
df_list = []

if not PROCESSED_DIR.exists():
    print(f"경로를 찾을 수 없습니다: {PROCESSED_DIR}")
else:
    parquet_files = list(PROCESSED_DIR.glob("*.parquet"))
    parquet_files = [f for f in parquet_files if "Master_Gait_Dataset" not in f.name and "raw_merged" not in f.name]

    print(f"총 {len(parquet_files)}개의 데이터 병합 파케이(Parquet) 파일 발견됨.\n")
    
    # 전체 파일 대상 구조 분해 및 검증 함수 반복 실행
    for p_file in parquet_files:
        df = verify_parquet_file(p_file) # 검증 함수가 df를 반환함
        display(df.columns.to_frame(index=False,name="columns"))
        df_list.append(df) # 🪣 2. 리스트 바구니에 로드된 df를 하나씩 가볍게 던져 넣음!
        break



총 4개의 데이터 병합 파케이(Parquet) 파일 발견됨.


🔍 [분석 시작]: ACLD.parquet
✅ Data Shape: 510,051 Rows × 693 Columns
✅ 필수 메타데이터(5종) 정상 마운트
👤 포함된 피험자 종류(40명): ['ACLD9', 'ACLD8', 'ACLD7', 'ACLD6', 'ACLD5']...
🏃‍♂️ 포함된 보행속도 분포: ['fast', 'normal', 'slow']
⚠️ 결측치(NaN)가 포함된 피처 개수: 70개


,columns
0,time_ms
1,orientation_0
2,orientation_1
3,orientation_2
4,orientation_3
5,orientation_4
6,orientation_5
7,orientation_6
8,orientation_7
9,orientation_8


In [ ]:

    # 🔨 3. 루프가 모두 끝난 뒤, 바구니에 담긴 4개의 덩어리를 단 한 번의 연산으로 합체!
    print("전체 데이터를 메모리에서 하나로 결합하는 중...")
    merged_df = pd.concat(df_list, ignore_index=True)
    
    print(f"\n✅ 드디어 하나로 차곡차곡 쌓인 마스터 데이터프레임 완성!")
    display(merged_df.info())  # 결과 확인


In [5]:
display(merged_df.info())
display(merged_df.head())
display(merged_df.describe())
output_path = PROCESSED_DIR / "raw_merged.parquet"

merged_df.to_parquet(output_path, index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1437356 entries, 0 to 1437355
Columns: 693 entries, time_ms to sensorOrientation_27
dtypes: float64(688), object(5)
memory usage: 7.4+ GB


None

,time_ms,orientation_0,orientation_1,orientation_2,orientation_3,orientation_4,orientation_5,orientation_6,orientation_7,orientation_8,orientation_9,orientation_10,orientation_11,orientation_12,orientation_13,orientation_14,orientation_15,orientation_16,orientation_17,orientation_18,orientation_19,orientation_20,orientation_21,orientation_22,orientation_23,orientation_24,orientation_25,orientation_26,orientation_27,orientation_28,orientation_29,orientation_30,orientation_31,orientation_32,orientation_33,orientation_34,orientation_35,orientation_36,orientation_37,orientation_38,orientation_39,orientation_40,orientation_41,orientation_42,orientation_43,orientation_44,orientation_45,orientation_46,orientation_47,orientation_48,orientation_49,orientation_50,orientation_51,orientation_52,orientation_53,orientation_54,orientation_55,orientation_56,orientation_57,orientation_58,orientation_59,orientation_60,orientation_61,orientation_62,orientation_63,orientation_64,orientation_65,orientation_66,orientation_67,orientation_68,orientation_69,orientation_70,orientation_71,orientation_72,orientation_73,orientation_74,orientation_75,orientation_76,orientation_77,orientation_78,orientation_79,orientation_80,orientation_81,orientation_82,orientation_83,orientation_84,orientation_85,orientation_86,orientation_87,orientation_88,orientation_89,orientation_90,orientation_91,position_0,position_1,position_2,position_3,position_4,position_5,position_6,position_7,position_8,position_9,position_10,position_11,position_12,position_13,position_14,position_15,position_16,position_17,position_18,position_19,position_20,position_21,position_22,position_23,position_24,position_25,position_26,position_27,position_28,position_29,position_30,position_31,position_32,position_33,position_34,position_35,position_36,position_37,position_38,position_39,position_40,position_41,position_42,position_43,position_44,position_45,position_46,position_47,position_48,position_49,position_50,position_51,position_52,position_53,position_54,position_55,position_56,position_57,position_58,position_59,position_60,position_61,position_62,position_63,position_64,position_65,position_66,position_67,position_68,velocity_0,velocity_1,velocity_2,velocity_3,velocity_4,velocity_5,velocity_6,velocity_7,velocity_8,velocity_9,velocity_10,velocity_11,velocity_12,velocity_13,velocity_14,velocity_15,velocity_16,velocity_17,velocity_18,velocity_19,velocity_20,velocity_21,velocity_22,velocity_23,velocity_24,velocity_25,velocity_26,velocity_27,velocity_28,velocity_29,velocity_30,velocity_31,velocity_32,velocity_33,velocity_34,velocity_35,velocity_36,velocity_37,velocity_38,velocity_39,velocity_40,velocity_41,velocity_42,velocity_43,velocity_44,velocity_45,velocity_46,velocity_47,velocity_48,velocity_49,velocity_50,velocity_51,velocity_52,velocity_53,velocity_54,velocity_55,velocity_56,velocity_57,velocity_58,velocity_59,velocity_60,velocity_61,velocity_62,velocity_63,velocity_64,velocity_65,velocity_66,velocity_67,velocity_68,acceleration_0,acceleration_1,acceleration_2,acceleration_3,acceleration_4,acceleration_5,acceleration_6,acceleration_7,acceleration_8,acceleration_9,acceleration_10,acceleration_11,acceleration_12,acceleration_13,acceleration_14,acceleration_15,acceleration_16,acceleration_17,acceleration_18,acceleration_19,acceleration_20,acceleration_21,acceleration_22,acceleration_23,acceleration_24,acceleration_25,acceleration_26,acceleration_27,acceleration_28,acceleration_29,acceleration_30,acceleration_31,acceleration_32,acceleration_33,acceleration_34,acceleration_35,acceleration_36,acceleration_37,acceleration_38,acceleration_39,acceleration_40,acceleration_41,acceleration_42,acceleration_43,acceleration_44,acceleration_45,acceleration_46,acceleration_47,acceleration_48,acceleration_49,acceleration_50,acceleration_51,acceleration_52,acceleration_53,acceleration_54,acceleration_55,acceleration_56,acceleration_57,acceleration_58,acceleration_59,acceleration_60,acceleration_61,acc

KeyboardInterrupt: 

In [ ]:
import pandas as pd



mergerd_df.column

NameError: name 'mergerd_df' is not defined

In [ ]:


# 전체 행 개수 대비 결측치 비율(%) 계산
na_percent = (df.isna().mean() * 100)
bad_cols_percent = na_percent[na_percent > 0].sort_values(ascending=False)

print("결측도 상위 불량 컬럼 목록(%):")
display(bad_cols_percent.head(100)) # 상위 20개만 보기
